In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

SEMENTE = 42
np.random.seed(SEMENTE)

# Caminho do dataset no seu projeto
ARQUIVO_DATASET = "../metadados/etapa 2/dataset_rotulado_periodo_b.csv"
print("Ambiente configurado com sucesso!")

Ambiente configurado com sucesso!


In [2]:
# 1. Carregar dataset
df = pd.read_csv(ARQUIVO_DATASET)

# Ordenação temporal estrita se houver a coluna timestamp
if "timestamp" in df.columns:
    df = df.sort_values("timestamp").reset_index(drop=True)

# 2. Divisão cronológica dos dados
total_registos = len(df)
corte_treino = int(total_registos * 0.70)
corte_val = int(total_registos * 0.85)

df_treino = df.iloc[:corte_treino].copy()
df_val = df.iloc[corte_treino:corte_val].copy()
df_teste = df.iloc[corte_val:].copy()

# 3. Definição das features oficiais e alvo
FEATURES_OFICIAIS = [
    "z_robusto", "aumento_pct", "jitter_relativo", "perda_pct",
    "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"
]
TARGET = "classe"
CLASSES_ORDEM = ["OK", "RISCO", "FALHA"]

X_train, y_train = df_treino[FEATURES_OFICIAIS], df_treino[TARGET]
X_val, y_val = df_val[FEATURES_OFICIAIS], df_val[TARGET]
X_test, y_test = df_teste[FEATURES_OFICIAIS], df_teste[TARGET]

print("--- DIVISÃO TEMPORAL DOS DADOS ---")
print(f"• Treino (70%):    {len(X_train):,} amostras")
print(f"• Validação (15%): {len(X_val):,} amostras")
print(f"• Teste (15%):     {len(X_test):,} amostras")

--- DIVISÃO TEMPORAL DOS DADOS ---
• Treino (70%):    71,364 amostras
• Validação (15%): 15,292 amostras
• Teste (15%):     15,293 amostras


In [3]:
# Definir a grade de busca de hiperparâmetros
parametros = {
    "n_estimators": [100, 200],
    "max_depth": [6, 12, None],
    "min_samples_leaf": [5, 50],
    "max_features": ["sqrt"],
    "class_weight": [None, "balanced_subsample"]
}

# Configurar o GridSearchCV com validação cruzada nos dados de treino
grid = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=SEMENTE,
        n_jobs=1
    ),
    param_grid=parametros,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

print("GridSearchCV configurado para o Random Forest!")

GridSearchCV configurado para o Random Forest!


In [4]:
print("Iniciando a busca de hiperparâmetros (GridSearch)...")
grid.fit(X_train, y_train)

print("\n--- MELHORES PARÂMETROS ENCONTRADOS ---")
print(grid.best_params_)

print(f"\nMelhor F1-Macro na Validação Cruzada (Treino): {grid.best_score_:.4f}")

Iniciando a busca de hiperparâmetros (GridSearch)...

--- MELHORES PARÂMETROS ENCONTRADOS ---
{'class_weight': 'balanced_subsample', 'max_depth': 12, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 200}

Melhor F1-Macro na Validação Cruzada (Treino): 0.9990


In [5]:
# Obter o melhor modelo treinado
modelo_rf = grid.best_estimator_

# Previsão no bloco de validação
y_val_pred = modelo_rf.predict(X_val)

cm = confusion_matrix(y_val, y_val_pred, labels=CLASSES_ORDEM)
df_cm = pd.DataFrame(
    cm,
    index=[f"Real_{c}" for c in CLASSES_ORDEM],
    columns=[f"Pred_{c}" for c in CLASSES_ORDEM]
)

print("--- MATRIZ DE CONFUSÃO 3x3 (VALIDAÇÃO) ---")
print(df_cm)

print("\n--- RELATÓRIO DE CLASSIFICAÇÃO NA VALIDAÇÃO ---")
print(classification_report(y_val, y_val_pred, labels=CLASSES_ORDEM, digits=4))

f1_macro_val = f1_score(y_val, y_val_pred, average="macro")
print(f"\n>>> F1-SCORE MACRO NO RANDOM FOREST (VALIDAÇÃO): {f1_macro_val:.4f} <<<")

--- MATRIZ DE CONFUSÃO 3x3 (VALIDAÇÃO) ---
            Pred_OK  Pred_RISCO  Pred_FALHA
Real_OK       11925           2           0
Real_RISCO        8        3300           0
Real_FALHA        0           0          57

--- RELATÓRIO DE CLASSIFICAÇÃO NA VALIDAÇÃO ---
              precision    recall  f1-score   support

          OK     0.9993    0.9998    0.9996     11927
       RISCO     0.9994    0.9976    0.9985      3308
       FALHA     1.0000    1.0000    1.0000        57

    accuracy                         0.9993     15292
   macro avg     0.9996    0.9991    0.9994     15292
weighted avg     0.9993    0.9993    0.9993     15292


>>> F1-SCORE MACRO NO RANDOM FOREST (VALIDAÇÃO): 0.9994 <<<


In [6]:
# 4. Definir a grade de hiperparâmetros
parametros = {
    "n_estimators": [100, 200],
    "max_depth": [6, 12, None],
    "min_samples_leaf": [5, 50],
    "max_features": ["sqrt"],
    "class_weight": [None, "balanced_subsample"]
}

In [ ]:
# Salvar o modelo Random Forest no mesmo diretório local
joblib.dump(modelo_rf, "modelos/random_forest_oficial.joblib")
print("Modelo Random Forest salvo com sucesso como 'random_forest_oficial.joblib'!")

Modelo Random Forest salvo com sucesso como 'random_forest_oficial.joblib'!
